# Early VLM data exploration

Notebook supplied by Muhammad Bin Tariq; see the repository provenance for the accompanying script implementation. Saved outputs and runtime metadata have been cleared for publication; recorded results and their original cell indices are documented in the project README. The original source notebook is unchanged.

This is an experimental workflow, not a claim of a newly reproduced run. Review configuration and GPU/model access requirements before execution.


# Environment Setup

In [ ]:
import os
import math
import random
from dataclasses import dataclass
from typing import List, Dict, Tuple, Optional

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import torchvision
from torchvision.datasets import CIFAR10

from datasets import load_dataset

from transformers import (
    CLIPImageProcessor,
    CLIPVisionModel,
    AutoTokenizer,
    AutoModelForCausalLM,
)

import matplotlib.pyplot as plt

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# Part A: Continuous Connector VLMs

## Task A-C0

In [ ]:
# ============================================================
# A-C0 constants
# ============================================================

CIFAR10_CLASSES = [
    "airplane", "automobile", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck"
]

# Helpful semantic groups for VQA templates
VEHICLE_CLASSES = {"airplane", "automobile", "ship", "truck"}
LIVING_CLASSES = {"bird", "cat", "deer", "dog", "frog", "horse"}
ANIMAL_CLASSES = {"bird", "cat", "deer", "dog", "frog", "horse"}
FLYING_CLASSES = {"airplane", "bird"}

N_TRAIN_PER_CLASS = 1000
N_TEST_PER_CLASS = 200

### Task A-C0.1

In [ ]:
# ============================================================
# A-C0.1 Load CIFAR-10 and stratified subsets
# ============================================================

train_full = CIFAR10(root="./data", train=True, download=True)
test_full = CIFAR10(root="./data", train=False, download=True)

def make_stratified_indices(
    targets: List[int],
    n_per_class: int,
    num_classes: int = 10,
    seed: int = 42,
) -> List[int]:
    rng = np.random.default_rng(seed)
    targets = np.array(targets)

    selected = []
    for cls in range(num_classes):
        cls_indices = np.where(targets == cls)[0]
        chosen = rng.choice(cls_indices, size=n_per_class, replace=False)
        selected.extend(chosen.tolist())

    rng.shuffle(selected)
    return selected

train_indices = make_stratified_indices(
    train_full.targets,
    n_per_class=N_TRAIN_PER_CLASS,
    seed=SEED,
)

test_indices = make_stratified_indices(
    test_full.targets,
    n_per_class=N_TEST_PER_CLASS,
    seed=SEED,
)

print("Train subset size:", len(train_indices))
print("Test subset size:", len(test_indices))

def class_counts(targets, indices):
    arr = np.array(targets)[indices]
    return {CIFAR10_CLASSES[c]: int((arr == c).sum()) for c in range(10)}

print("Train class counts:", class_counts(train_full.targets, train_indices))
print("Test class counts:", class_counts(test_full.targets, test_indices))

In [ ]:
# ============================================================
# A-C0.1 CLIP preprocessing
# ============================================================

clip_processor = CLIPImageProcessor.from_pretrained("openai/clip-vit-base-patch32")

print("CLIP image mean:", clip_processor.image_mean)
print("CLIP image std: ", clip_processor.image_std)

imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std = [0.229, 0.224, 0.225]

print("ImageNet mean:", imagenet_mean)
print("ImageNet std: ", imagenet_std)

print("CLIP mean differs from ImageNet:",
      clip_processor.image_mean != imagenet_mean)
print("CLIP std differs from ImageNet:",
      clip_processor.image_std != imagenet_std)

In [ ]:
# ============================================================
# A-C0.1 Cache CLIP-preprocessed image tensors
# ============================================================

def preprocess_subset(dataset, indices, processor):
    pixel_values = []
    labels = []

    for idx in indices:
        image, label = dataset[idx]
        processed = processor(images=image, return_tensors="pt")
        pixel_values.append(processed["pixel_values"].squeeze(0))
        labels.append(label)

    pixel_values = torch.stack(pixel_values)
    labels = torch.tensor(labels, dtype=torch.long)
    return pixel_values, labels

train_pixels, train_labels = preprocess_subset(train_full, train_indices, clip_processor)
test_pixels, test_labels = preprocess_subset(test_full, test_indices, clip_processor)

print("train_pixels:", train_pixels.shape, train_pixels.dtype)
print("train_labels:", train_labels.shape)
print("test_pixels: ", test_pixels.shape, test_pixels.dtype)
print("test_labels: ", test_labels.shape)

In [ ]:
# ============================================================
# Optional: visualize a few preprocessed images
# ============================================================

def unnormalize_clip_image(x):
    mean = torch.tensor(clip_processor.image_mean).view(3, 1, 1)
    std = torch.tensor(clip_processor.image_std).view(3, 1, 1)
    x = x.cpu() * std + mean
    return x.clamp(0, 1)

fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for ax, idx in zip(axes.flat, range(10)):
    img = unnormalize_clip_image(train_pixels[idx]).permute(1, 2, 0)
    ax.imshow(img)
    ax.set_title(CIFAR10_CLASSES[train_labels[idx].item()])
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# A-C0.2 Synthetic caption dataset
# ============================================================

CAPTION_TEMPLATES = [
    "a photo of a {}.",
    "this image shows a {}.",
    "there is a {} in the image.",
    "the object shown is a {}.",
    "an image of a {}.",
    "the picture contains a {}.",
]

def build_caption_records(labels: torch.Tensor) -> List[Dict]:
    records = []
    for image_idx, label_id in enumerate(labels.tolist()):
        class_name = CIFAR10_CLASSES[label_id]
        template = CAPTION_TEMPLATES[image_idx % len(CAPTION_TEMPLATES)]
        caption = template.format(class_name)

        records.append({
            "image_idx": image_idx,
            "label_id": label_id,
            "class_name": class_name,
            "caption": caption,
        })
    return records

caption_train_records = build_caption_records(train_labels)

print("Number of caption records:", len(caption_train_records))
for r in caption_train_records[:8]:
    print(r)

In [ ]:
# ============================================================
# A-C0.2 Synthetic VQA dataset
# ============================================================

VQA_TEMPLATE_NAMES = [
    "object_recognition",
    "class_presence",
    "vehicle_or_living",
    "can_fly",
    "is_animal",
]

def deterministic_wrong_class(true_label: int) -> int:
    # guaranteed different, deterministic, balanced enough for the toy setup
    return (true_label + 1) % 10

def build_vqa_records(labels: torch.Tensor) -> List[Dict]:
    records = []

    for image_idx, label_id in enumerate(labels.tolist()):
        class_name = CIFAR10_CLASSES[label_id]

        # 1. Object recognition
        records.append({
            "image_idx": image_idx,
            "label_id": label_id,
            "class_name": class_name,
            "template_id": 0,
            "template_name": "object_recognition",
            "question": "What object is shown?",
            "answer": class_name,
        })

        # 2. Class presence: alternate positive / negative queries
        if image_idx % 2 == 0:
            queried_label = label_id
            answer = "yes"
        else:
            queried_label = deterministic_wrong_class(label_id)
            answer = "no"

        queried_class = CIFAR10_CLASSES[queried_label]
        records.append({
            "image_idx": image_idx,
            "label_id": label_id,
            "class_name": class_name,
            "template_id": 1,
            "template_name": "class_presence",
            "question": f"Is there a {queried_class}?",
            "answer": answer,
            "queried_class": queried_class,
        })

        # 3. Vehicle or living thing
        answer = "vehicle" if class_name in VEHICLE_CLASSES else "living thing"
        records.append({
            "image_idx": image_idx,
            "label_id": label_id,
            "class_name": class_name,
            "template_id": 2,
            "template_name": "vehicle_or_living",
            "question": "Vehicle or living thing?",
            "answer": answer,
        })

        # 4. Can it fly?
        answer = "yes" if class_name in FLYING_CLASSES else "no"
        records.append({
            "image_idx": image_idx,
            "label_id": label_id,
            "class_name": class_name,
            "template_id": 3,
            "template_name": "can_fly",
            "question": "Can it fly?",
            "answer": answer,
        })

        # 5. Is this an animal?
        answer = "yes" if class_name in ANIMAL_CLASSES else "no"
        records.append({
            "image_idx": image_idx,
            "label_id": label_id,
            "class_name": class_name,
            "template_id": 4,
            "template_name": "is_animal",
            "question": "Is this an animal?",
            "answer": answer,
        })

    return records

vqa_train_records = build_vqa_records(train_labels)
vqa_val_records = build_vqa_records(test_labels)

print("Train VQA pairs:", len(vqa_train_records))
print("Val VQA pairs:", len(vqa_val_records))

for r in vqa_train_records[:10]:
    print(r["question"], "->", r["answer"])

In [ ]:
from collections import Counter, defaultdict

def answer_counts_by_template(records):
    counts = defaultdict(Counter)
    for r in records:
        counts[r["template_name"]][r["answer"]] += 1
    return counts

for template, counts in answer_counts_by_template(vqa_train_records).items():
    print(template, dict(counts))

In [ ]:
# ============================================================
# Dataset wrappers
# ============================================================

class CaptionDataset(Dataset):
    def __init__(self, pixel_values: torch.Tensor, records: List[Dict]):
        self.pixel_values = pixel_values
        self.records = records

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        r = self.records[idx]
        return {
            "pixel_values": self.pixel_values[r["image_idx"]],
            "caption": r["caption"],
            "label_id": r["label_id"],
            "class_name": r["class_name"],
        }


class VQADataset(Dataset):
    def __init__(self, pixel_values: torch.Tensor, records: List[Dict]):
        self.pixel_values = pixel_values
        self.records = records

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        r = self.records[idx]
        return {
            "pixel_values": self.pixel_values[r["image_idx"]],
            "question": r["question"],
            "answer": r["answer"],
            "label_id": r["label_id"],
            "class_name": r["class_name"],
            "template_id": r["template_id"],
            "template_name": r["template_name"],
        }

caption_train_ds = CaptionDataset(train_pixels, caption_train_records)
vqa_train_ds = VQADataset(train_pixels, vqa_train_records)
vqa_val_ds = VQADataset(test_pixels, vqa_val_records)

print("caption_train_ds:", len(caption_train_ds))
print("vqa_train_ds:", len(vqa_train_ds))
print("vqa_val_ds:", len(vqa_val_ds))

In [ ]:
# ============================================================
# A-C0.3 Load CLIP vision encoder
# ============================================================

clip_vision = CLIPVisionModel.from_pretrained("openai/clip-vit-base-patch32")
clip_vision.to(device)
clip_vision.eval()

for p in clip_vision.parameters():
    p.requires_grad = False

# Sanity check on one mini-batch
with torch.no_grad():
    sample_pixels = train_pixels[:2].to(device)
    clip_out = clip_vision(pixel_values=sample_pixels)
    last_hidden = clip_out.last_hidden_state

print("CLIP last_hidden_state shape:", last_hidden.shape)
print("All CLIP params frozen:", all(not p.requires_grad for p in clip_vision.parameters()))

# Extract spatial patch tokens only
patch_tokens = last_hidden[:, 1:, :]
print("Patch token shape after discarding CLS:", patch_tokens.shape)

In [ ]:
# ============================================================
# A-C0.3 Load SmolLM2
# ============================================================

LM_NAME = "HuggingFaceTB/SmolLM2-360M-Instruct"

tokenizer = AutoTokenizer.from_pretrained(LM_NAME)
lm = AutoModelForCausalLM.from_pretrained(
    LM_NAME,
    torch_dtype=torch.float16,
)
lm.to(device)
lm.eval()

# For decoder-only batching
tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("LM hidden size:", lm.config.hidden_size)
print("LM vocab size:", lm.config.vocab_size)
print("Embedding table shape:", lm.get_input_embeddings().weight.shape)
print("Tokenizer bos token:", tokenizer.bos_token, tokenizer.bos_token_id)
print("Tokenizer eos token:", tokenizer.eos_token, tokenizer.eos_token_id)
print("Tokenizer pad token:", tokenizer.pad_token, tokenizer.pad_token_id)

In [ ]:
# ============================================================
# A-C0.3 Load Alpaca replay subset
# ============================================================

alpaca = load_dataset("tatsu-lab/alpaca", split="train")
alpaca = alpaca.shuffle(seed=SEED).select(range(1000))

print("Alpaca subset size:", len(alpaca))
print(alpaca[0])

In [ ]:
# ============================================================
# A-C0.3 Alpaca formatting for text-only replay
# ============================================================

def to_id_list(tokenized):
    """
    Convert the output of apply_chat_template into a plain list[int],
    whether it returns a list or a BatchEncoding.
    """
    if isinstance(tokenized, dict) or hasattr(tokenized, "keys"):
        tokenized = tokenized["input_ids"]

    # In case a tensor is returned in a future call/version
    if isinstance(tokenized, torch.Tensor):
        tokenized = tokenized.tolist()

    # If there is an accidental batch dimension, remove it
    if len(tokenized) > 0 and isinstance(tokenized[0], list):
        tokenized = tokenized[0]

    return list(tokenized)


def make_user_content(example: Dict) -> str:
    instruction = example["instruction"].strip()
    inp = example["input"].strip()

    if inp:
        return f"{instruction}\n\nInput:\n{inp}"
    return instruction


def build_chat_text_and_labels(example: Dict):
    """
    Build one full chat sequence and labels for assistant-response-only loss.
    """
    user_content = make_user_content(example)
    assistant_content = example["output"].strip()

    # Prompt ending right before assistant answer begins
    prompt_messages = [
        {"role": "user", "content": user_content},
    ]
    prompt_ids = tokenizer.apply_chat_template(
        prompt_messages,
        tokenize=True,
        add_generation_prompt=True,
    )
    prompt_ids = to_id_list(prompt_ids)

    # Full conversation including answer
    full_messages = [
        {"role": "user", "content": user_content},
        {"role": "assistant", "content": assistant_content},
    ]
    full_ids = tokenizer.apply_chat_template(
        full_messages,
        tokenize=True,
        add_generation_prompt=False,
    )
    full_ids = to_id_list(full_ids)

    labels = [-100] * len(prompt_ids) + full_ids[len(prompt_ids):]

    return {
        "input_ids": full_ids,
        "labels": labels,
        "user_content": user_content,
        "assistant_content": assistant_content,
    }

alpaca_encoded = [build_chat_text_and_labels(ex) for ex in alpaca]

print("Type check:", type(alpaca_encoded[0]["input_ids"]))
print("First few token ids:", alpaca_encoded[0]["input_ids"][:10])
print("Input length:", len(alpaca_encoded[0]["input_ids"]))
print("Number of supervised tokens:",
      sum(x != -100 for x in alpaca_encoded[0]["labels"]))

In [ ]:
# ============================================================
# A-C0.3 Text replay collator
# ============================================================

def collate_text_replay(batch: List[Dict]):
    max_len = max(len(x["input_ids"]) for x in batch)

    input_ids = []
    labels = []
    attention_mask = []

    for x in batch:
        ids = x["input_ids"]
        labs = x["labels"]
        pad_len = max_len - len(ids)

        # left padding because decoder-only batched generation later also needs left padding
        input_ids.append([tokenizer.pad_token_id] * pad_len + ids)
        labels.append([-100] * pad_len + labs)
        attention_mask.append([0] * pad_len + [1] * len(ids))

    return {
        "input_ids": torch.tensor(input_ids, dtype=torch.long),
        "labels": torch.tensor(labels, dtype=torch.long),
        "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
    }

class AlpacaReplayDataset(Dataset):
    def __init__(self, encoded_records):
        self.encoded_records = encoded_records

    def __len__(self):
        return len(self.encoded_records)

    def __getitem__(self, idx):
        return self.encoded_records[idx]

alpaca_replay_ds = AlpacaReplayDataset(alpaca_encoded)
alpaca_replay_loader = DataLoader(
    alpaca_replay_ds,
    batch_size=8,
    shuffle=False,
    collate_fn=collate_text_replay,
)

batch = next(iter(alpaca_replay_loader))
for k, v in batch.items():
    print(k, v.shape, v.dtype)

In [ ]:
# ============================================================
# A-C0.3 Compute baseline PPL0
# ============================================================

@torch.no_grad()
def compute_response_ppl(model, dataloader, device):
    model.eval()

    total_nll = 0.0
    total_tokens = 0

    for batch in dataloader:
        batch = {k: v.to(device) for k, v in batch.items()}

        outputs = model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
            labels=batch["labels"],
        )

        # Hugging Face causal LM loss is already mean over non--100 labels
        n_tokens = (batch["labels"] != -100).sum().item()
        total_nll += outputs.loss.item() * n_tokens
        total_tokens += n_tokens

    mean_nll = total_nll / total_tokens
    ppl = math.exp(mean_nll)
    return mean_nll, ppl

ppl0_nll, PPL0 = compute_response_ppl(lm, alpaca_replay_loader, device)

print(f"Baseline Alpaca response NLL: {ppl0_nll:.4f}")
print(f"Baseline PPL0: {PPL0:.4f}")

# Gemini

In [ ]:
import torch
import torchvision
import torchvision.transforms as transforms
from transformers import CLIPVisionModel, CLIPImageProcessor
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
import numpy as np

# Device setup (Colab Pro+ will likely give you an A100 or V100)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

def load_models():
    print("Loading CLIP Vision Encoder...")
    # Using the exact baseline model specified
    clip_id = "openai/clip-vit-base-patch32"
    clip_processor = CLIPImageProcessor.from_pretrained(clip_id)
    clip_model = CLIPVisionModel.from_pretrained(clip_id).to(device)

    # Freeze CLIP completely [cite: 277]
    clip_model.eval()
    for param in clip_model.parameters():
        param.requires_grad = False

    print("Loading SmolLM2-360M...")
    lm_id = "HuggingFaceTB/SmolLM2-360M-Instruct"
    lm_tokenizer = AutoTokenizer.from_pretrained(lm_id)
    # Left-padding is crucial for decoder-only batched generation later [cite: 365]
    lm_tokenizer.padding_side = "left"
    if lm_tokenizer.pad_token is None:
        lm_tokenizer.pad_token = lm_tokenizer.eos_token

    # Load in float16 as mandated [cite: 278]
    lm_model = AutoModelForCausalLM.from_pretrained(
        lm_id,
        torch_dtype=torch.float16
    ).to(device)

    return clip_processor, clip_model, lm_tokenizer, lm_model

clip_processor, clip_model, lm_tokenizer, lm_model = load_models()

In [ ]:
def get_visual_embeddings(images, processor, vision_model):
    """
    images: list of PIL Images or raw tensors
    Returns: Tensor of shape (batch_size, 49, 768)
    """
    # The processor handles resizing to 224x224 and CLIP normalization [cite: 268]
    inputs = processor(images=images, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = vision_model(**inputs)
        # Extract last hidden state: (batch, 50, 768)
        last_hidden_state = outputs.last_hidden_state

        # Slice off the CLS token at index 0
        spatial_patches = last_hidden_state[:, 1:, :]

    return spatial_patches

In [ ]:
def load_and_stratify_data():
    # Load raw CIFAR-10
    train_ds = torchvision.datasets.CIFAR10(root='./data', train=True, download=True)
    test_ds = torchvision.datasets.CIFAR10(root='./data', train=False, download=True)

    # Stratify train: 1000 per class (Seed 42) [cite: 267]
    np.random.seed(42)
    train_indices = []
    for class_idx in range(10):
        idx = np.where(np.array(train_ds.targets) == class_idx)[0]
        train_indices.extend(np.random.choice(idx, 1000, replace=False))

    # Stratify test: 200 per class [cite: 267]
    test_indices = []
    for class_idx in range(10):
        idx = np.where(np.array(test_ds.targets) == class_idx)[0]
        test_indices.extend(np.random.choice(idx, 200, replace=False))

    train_subset = torch.utils.data.Subset(train_ds, train_indices)
    test_subset = torch.utils.data.Subset(test_ds, test_indices)

    print(f"CIFAR-10 Train size: {len(train_subset)}")
    print(f"CIFAR-10 Test size: {len(test_subset)}")

    # Load exactly 1000 Alpaca examples for replay [cite: 279]
    alpaca_ds = load_dataset("tatsu-lab/alpaca", split="train[:1000]")
    print(f"Alpaca Replay size: {len(alpaca_ds)}")

    return train_subset, test_subset, alpaca_ds

cifar_train, cifar_test, alpaca = load_and_stratify_data()

In [ ]:
import random
from torch.utils.data import Dataset, DataLoader
import math

# Define the semantics for CIFAR-10 classes
cifar_classes = ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']
class_attributes = {
    'airplane': {'type': 'vehicle', 'fly': 'yes', 'animal': 'no'},
    'automobile': {'type': 'vehicle', 'fly': 'no', 'animal': 'no'},
    'bird': {'type': 'living', 'fly': 'yes', 'animal': 'yes'},
    'cat': {'type': 'living', 'fly': 'no', 'animal': 'yes'},
    'deer': {'type': 'living', 'fly': 'no', 'animal': 'yes'},
    'dog': {'type': 'living', 'fly': 'no', 'animal': 'yes'},
    'frog': {'type': 'living', 'fly': 'no', 'animal': 'yes'},
    'horse': {'type': 'living', 'fly': 'no', 'animal': 'yes'},
    'ship': {'type': 'vehicle', 'fly': 'no', 'animal': 'no'},
    'truck': {'type': 'vehicle', 'fly': 'no', 'animal': 'no'}
}

def generate_vqa_dataset(vision_dataset):
    vqa_data = []

    for idx in range(len(vision_dataset)):
        # We don't need the image tensor here, just the label for string generation
        _, label_idx = vision_dataset[idx]
        true_class = cifar_classes[label_idx]
        attrs = class_attributes[true_class]

        # 1. Recognition
        vqa_data.append({"image_idx": idx, "q": "What object is shown?", "a": true_class})

        # 2. Binary (50% chance of asking about the true class, 50% fake class)
        ask_true = random.choice([True, False])
        query_class = true_class if ask_true else random.choice([c for c in cifar_classes if c != true_class])
        vqa_data.append({"image_idx": idx, "q": f"Is there a {query_class}?", "a": "yes" if ask_true else "no"})

        # 3. Abstraction
        vqa_data.append({"image_idx": idx, "q": "Vehicle or living thing?", "a": attrs['type']})

        # 4. Attribute
        vqa_data.append({"image_idx": idx, "q": "Can it fly?", "a": attrs['fly']})

        # 5. Category
        vqa_data.append({"image_idx": idx, "q": "Is this an animal?", "a": attrs['animal']})

    return vqa_data

print("Generating VQA pairs...")
# 10,000 images * 5 templates = 50,000 train pairs
train_vqa = generate_vqa_dataset(cifar_train)
# 2,000 images * 5 templates = 10,000 val pairs
val_vqa = generate_vqa_dataset(cifar_test)

print(f"Train VQA pairs: {len(train_vqa)}")
print(f"Val VQA pairs: {len(val_vqa)}")

In [ ]:
from tqdm import tqdm

def format_alpaca(example):
    # Standard Alpaca instruction formatting
    if example.get("input", ""):
        return f"Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.\n\n### Instruction:\n{example['instruction']}\n\n### Input:\n{example['input']}\n\n### Response:\n{example['output']}"
    return f"Below is an instruction that describes a task. Write a response that appropriately completes the request.\n\n### Instruction:\n{example['instruction']}\n\n### Response:\n{example['output']}"

def compute_baseline_ppl(model, tokenizer, dataset, batch_size=8):
    model.eval()
    total_loss = 0.0
    total_tokens = 0

    # Format texts
    texts = [format_alpaca(ex) for ex in dataset]

    print("Computing Baseline Perplexity (PPL_0)...")
    with torch.no_grad():
        for i in tqdm(range(0, len(texts), batch_size)):
            batch_texts = texts[i:i+batch_size]
            inputs = tokenizer(batch_texts, return_tensors="pt", padding=True, truncation=True, max_length=512).to(device)

            # --- THE FIX ---
            # Clone the input IDs to use as labels
            labels = inputs["input_ids"].clone()
            # Set the label for all padding tokens to -100 so they are ignored in the loss calculation
            labels[inputs["attention_mask"] == 0] = -100

            outputs = model(**inputs, labels=labels)

            total_loss += outputs.loss.item() * len(batch_texts)

    mean_loss = total_loss / len(texts)
    ppl = math.exp(mean_loss)
    return ppl

# Make sure to run this before any training starts!
ppl_0 = compute_baseline_ppl(lm_model, lm_tokenizer, alpaca)
print(f"\nBaseline PPL_0: {ppl_0:.4f}")